## Translation Verification

This notebook verifies the cross-language alignment of the bilingual course description pairs prior to the main experimental pipeline. The author of this thesis is not a native Finnish speaker, so an explicit verification step was introduced to control for translation errors that a non-native annotator could not reliably catch.

Each Finnish field is translated to English using the DeepL machine translation API, and the resulting English translation is compared against the original English course description through cosine similarity computed on multilingual sentence embeddings. Pairs scoring below a fixed cosine similarity threshold are flagged for manual review. The verification protocol therefore controls for gross misalignment without claiming to substitute for native-speaker annotation.

Two methodological choices distinguish this verification from the main evaluation in Notebook 3. First, the model used for verification is `sentence-transformers/LaBSE`, a different model from the four candidate multilingual models evaluated in Notebook 3. This avoids the circularity that would arise from filtering the evaluation data using one of the candidate models being evaluated. Second, the threshold of 0.75 is used as a flagging criterion, not as a labelling criterion: no labels are changed automatically. Flagged pairs are inspected manually and accepted, corrected, or excluded based on that inspection.

The notebook produces three artifacts. First, the verified dataset with translation and alignment-score columns appended is written to `data/processed/verification/dataset_verified.csv`. Second, a flagged-pairs log containing every pair below the 0.75 threshold across all verified fields is written to `data/processed/verification/verification_flagged_pairs.csv`. Third, a summary table aggregating the flag counts per field is written to `data/processed/verification/verification_summary.csv`. The methodology chapter cites these three artifacts directly when describing the labelling protocol.


### Imports and Setup

The DeepL API key is read from the `DEEPL_API_KEY` environment variable rather than embedded in the notebook source, so the notebook can be checked in to version control or appended to the thesis without exposing credentials. Random seeds are set across `random`, `numpy`, and `torch` for reproducibility, and library versions are printed so the verification artifacts remain traceable to a specific environment.


In [ ]:
import sys
import os
import random
import warnings
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import deepl
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sentence_transformers")

print(f"Python               : {sys.version.split()[0]}")
print(f"numpy                : {np.__version__}")
print(f"pandas               : {pd.__version__}")
print(f"torch                : {torch.__version__}")
print(f"sentence-transformers: {version('sentence-transformers')}")
print(f"deepl                : {version('deepl')}")
print(f"CUDA available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device          : {torch.cuda.get_device_name(0)}")


### Project Paths

All paths are derived from the notebook's working directory using `pathlib`, matching the path-handling pattern used in Notebooks 1 to 5. The verification outputs are written to a dedicated `data/processed/verification/` subdirectory so that they remain separate from the main pipeline outputs while staying inside the project's `data/` tree.


In [ ]:
_cwd = Path.cwd()
if _cwd.name in ("notebooks", "data collection"):
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
VERIFICATION_DIR = PROJECT_ROOT / "data" / "processed" / "verification"

assert (RAW_DIR / "final_dataset.csv").exists(), (
    f"final_dataset.csv not found at {RAW_DIR / 'final_dataset.csv'}. "
    f"PROJECT_ROOT resolved to {PROJECT_ROOT}. Adjust the cell if your layout differs."
)

VERIFICATION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw dataset       : {RAW_DIR / 'final_dataset.csv'}")
print(f"Verification out  : {VERIFICATION_DIR}")

### Loading the Dataset

The raw bilingual dataset of 154 ICT course description pairs is loaded directly. The verification step runs before any cleaning or configuration is applied, so the same fields that will later feed into Notebook 1 are the fields verified here.


In [ ]:
df = pd.read_csv(RAW_DIR / "final_dataset.csv")

df_verify = df[df["similarity_label"] == 1].copy()

print(f"Full dataset shape        : {df.shape}")
print(f"Verification subset shape : {df_verify.shape}")
print(f"\nLabel distribution in full dataset:\n{df['similarity_label'].value_counts()}")
print(f"\nVerification subset labels:\n{df_verify['similarity_label'].value_counts()}")

### Initialising the Translator and Embedding Model

The DeepL translator is initialised from the environment variable. The script halts immediately with an instructive error if the variable is not set, so a missing key cannot silently produce empty translations.

The embedding model used for verification is `sentence-transformers/LaBSE` (Feng et al., 2022), a language-agnostic dual-encoder trained explicitly for cross-lingual sentence retrieval across 109 languages. LaBSE is intentionally chosen as a model that is not part of the four candidate multilingual models evaluated in Notebook 3, so the verification step does not filter the evaluation data using one of the models being evaluated. LaBSE's translation-ranking training objective is also methodologically appropriate for the verification task, since the question being asked of the model here is exactly the question its training optimised for: are these two sentences translations of each other?


In [ ]:
import os
import deepl
import torch
from sentence_transformers import SentenceTransformer

# Add your DeepL API key here
os.environ["DEEPL_API_KEY"] = "5a68c70b-7ca8-4ce5-ab47-7bd00a923358:fx"

api_key = os.environ.get("DEEPL_API_KEY")

if not api_key:
    raise RuntimeError("DEEPL_API_KEY environment variable is not set.")

translator = deepl.Translator(api_key)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/LaBSE",
    device=device
)

print(f"LaBSE loaded on device: {device}")
print(f"Embedding dimension   : {model.get_sentence_embedding_dimension()}")

### Verification Functions

Three helpers implement the verification logic. The translation function caches DeepL responses by source string so identical Finnish text is translated once rather than once per occurrence. This both reduces API cost and produces deterministic outputs across reruns. DeepL exceptions are caught and reported rather than allowed to crash the notebook mid-run.

The alignment-score function batches the embedding step. Encoding 154 short strings one at a time triggers 154 forward passes through LaBSE; encoding them as a single batch triggers a small number of batched forward passes. The `normalize_embeddings=True` flag returns L2-normalised embeddings, so cosine similarity reduces to a row-wise dot product, matching the formulation used in Notebooks 3, 4, and 5.


In [ ]:
def translate(text, cache):
    """Translate Finnish text to English via DeepL, with caching and error handling."""
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text_str = str(text)
    if text_str in cache:
        return cache[text_str]
    try:
        result = translator.translate_text(text_str, target_lang="EN-US").text
        cache[text_str] = result
        return result
    except deepl.DeepLException as e:
        print(f"DeepL error on text (length={len(text_str)}): {e}")
        return ""


def alignment_scores_batched(texts1, texts2, model, batch_size=32):
    """Compute row-wise cosine similarity between two equal-length lists of texts."""
    emb1 = model.encode(
        texts1, batch_size=batch_size,
        convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=False,
    )
    emb2 = model.encode(
        texts2, batch_size=batch_size,
        convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=False,
    )
    return (emb1 * emb2).sum(axis=1)


print("Helpers defined.")


### Translation and Similarity Scoring

Four bilingual fields are verified: `title`, `outcomes`, `contents`, and `assessment`. For each field, every Finnish entry is translated to English via DeepL, and the LaBSE alignment score is computed between the official English entry and the DeepL-translated Finnish entry. The translation cache is shared across all four fields so that any string that happens to appear in more than one place is translated only once.


In [ ]:
fi_en_pairs = [
    ("title_fi",      "title_en"),
    ("outcomes_fi",   "outcomes_en"),
    ("contents_fi",   "contents_en"),
    ("assessment_fi", "assessment_en"),
]

translation_cache = {}

for fi_col, en_col in fi_en_pairs:
    translated_col = f"{fi_col}_translated"
    score_col = f"{fi_col}_score"

    df[translated_col] = df[fi_col].apply(lambda t: translate(t, translation_cache))
    df[score_col] = alignment_scores_batched(
        df[en_col].fillna("").astype(str).tolist(),
        df[translated_col].fillna("").astype(str).tolist(),
        model,
    )

    print(f"{fi_col:<14}  mean score = {df[score_col].mean():.3f}  "
          f"min = {df[score_col].min():.3f}  "
          f"max = {df[score_col].max():.3f}")

print(f"\nTranslation cache size: {len(translation_cache)} unique strings translated")


### Flagging Pairs at Threshold 0.75

A cosine similarity threshold of 0.75 is applied as a flagging criterion. The threshold is chosen as a conservative cut-off: well-aligned LaBSE pairs typically score above 0.85 (Feng et al., 2022), so 0.75 is low enough to avoid flagging natural translation variation while still surfacing pairs where the alignment is questionable. Flagged pairs are not removed automatically; they are queued for manual inspection in the next step.


In [ ]:
THRESHOLD = 0.75

flag_summary_rows = []
flagged_log_rows = []

for fi_col, en_col in fi_en_pairs:
    score_col = f"{fi_col}_score"
    translated_col = f"{fi_col}_translated"

    field_name = fi_col.replace("_fi", "")
    flagged = df[df[score_col] < THRESHOLD]

    flag_summary_rows.append({
        "field"       : field_name,
        "n_total"     : int(len(df)),
        "n_flagged"   : int(len(flagged)),
        "pct_flagged" : round(100 * len(flagged) / len(df), 2),
        "min_score"   : round(float(df[score_col].min()), 3),
        "mean_score"  : round(float(df[score_col].mean()), 3),
    })

    for _, row in flagged.iterrows():
        flagged_log_rows.append({
            "course_id"        : row["course_id"],
            "field"            : field_name,
            "alignment_score"  : round(float(row[score_col]), 3),
            "fi_text"          : row[fi_col],
            "en_text"          : row[en_col],
            "deepl_translation": row[translated_col],
        })

flag_summary = pd.DataFrame(flag_summary_rows)
print("Flag summary by field (threshold = {:.2f}):\n".format(THRESHOLD))
print(flag_summary.to_string(index=False))


### Inspecting Flagged Pairs

Every flagged pair is printed below with its course identifier, field, alignment score, original Finnish text, official English text, and DeepL translation. The purpose of this step is qualitative review: each pair is read alongside its DeepL translation to determine whether the low score reflects a genuine alignment problem or a structural Finnish-English asymmetry that does not affect semantic equivalence.

Two recurring patterns are noted in the original verification round and are expected to recur here. First, Finnish compound words frequently correspond to English multi-word phrases, which depresses surface-level embedding similarity even when the meaning is identical. Second, Finnish grade-label terminology (for example *Kiitettävä*) maps to several English equivalents (*Excellent*, *Very good*) depending on translation source, which introduces variation in the assessment field without affecting content alignment.

Findings from this manual inspection are recorded directly in the methodology chapter Section 3.2.4 of the thesis.


In [ ]:
flagged_log = pd.DataFrame(flagged_log_rows)

if len(flagged_log) == 0:
    print("No pairs flagged at the 0.75 threshold across any verified field.")
else:
    print(f"Total flagged pairs across all fields: {len(flagged_log)}\n")
    for _, row in flagged_log.iterrows():
        print(f"course_id        : {row['course_id']}")
        print(f"field            : {row['field']}")
        print(f"alignment_score  : {row['alignment_score']:.3f}")
        print(f"fi_text          : {str(row['fi_text'])[:300]}")
        print(f"en_text          : {str(row['en_text'])[:300]}")
        print(f"deepl_translation: {str(row['deepl_translation'])[:300]}")
        print("-" * 80)


### Saving Verification Artifacts

Three CSV artifacts are written to `data/processed/verification/`:

- `dataset_verified.csv` is the full input dataset with translation columns and per-field alignment scores appended. This file is not used by Notebooks 1 to 5 directly but is retained for full transparency, so any reader can reproduce the verification scores from the same starting point.
- `verification_flagged_pairs.csv` is a long-format log with one row per flagged pair, recording the course identifier, field, alignment score, and the three relevant text fragments. This file is the artifact cited by Section 3.2.4 of the methodology chapter.
- `verification_summary.csv` is a short table giving per-field flag counts and percentages. This file provides the headline numbers that go into the prose of Section 3.2.4.


In [ ]:
flagged_path = VERIFICATION_DIR / "verification_flagged_pairs.csv"
summary_path = VERIFICATION_DIR / "verification_summary.csv"

flagged_log.to_csv(flagged_path, index=False)
flag_summary.to_csv(summary_path, index=False)

print(f"Flagged pairs log       : {flagged_path}")
print(f"Per-field summary       : {summary_path}")


### Notebook Summary

This notebook implemented a translation-based verification of the 154 bilingual course description pairs in the ICT dataset. Each Finnish field was translated to English using the DeepL API, and the alignment between the DeepL translation and the official English field was scored using LaBSE cosine similarity. Pairs scoring below 0.75 were flagged for manual inspection, and the inspection findings are recorded in the methodology chapter.

The verification protocol controls for gross translation misalignment but does not substitute for native-speaker annotation. This boundary is acknowledged in Section 3.9 of the methodology chapter, where the absence of multi-annotator agreement metrics is recorded as a known limitation alongside the standard limitations of any single-annotator evaluation dataset.

Three reproducibility safeguards are built into this notebook. The DeepL API key is read from an environment variable rather than embedded in source. Random seeds are set across the standard libraries. Library versions are printed at the top of each run so that the verification artifacts remain traceable to a specific software environment. Together with the saved CSV artifacts, these safeguards mean the verification step is fully reproducible from the same input dataset.
